In [13]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import time
from multiprocessing import Process
import multiprocessing
from random import randint

In [14]:
e_prob = 0.5
pkt_prob = 0.5
weight_prob= 0.5

In [15]:
h_vals_u1 = [0.1, 1]
h_bad = 0.5
h1_prob = [h_bad, (1-h_bad)]
B_max = 2
rmax = 1
B1 = np.arange(0, B_max+1, 1)
rem_bits_1 = np.arange(0, rmax+0.5, 0.5)
all_states = []
for i in B1:
    for i2 in B1:
        for k in rem_bits_1:
            for k2 in rem_bits_1:
                for m in h_vals_u1:
                    for m2 in h_vals_u1:
                                all_states.append((i, i2, k, k2, m,m2, n, n2))
all_states = np.array(all_states)
print(all_states.shape)

(324, 8)


In [16]:
type(all_states)

numpy.ndarray

In [5]:
def st_tr_wt(wt, wt_next):
    if wt == 1 and wt_next == 1:
        m = (1-pkt_prob) + (pkt_prob) * (weight_prob)
    elif wt == 1 and wt_next == 2:
        m = pkt_prob * (1 - weight_prob)
    elif wt == 2 and wt_next == 1:
        m = pkt_prob * weight_prob
    else:
        m = (1-pkt_prob) + pkt_prob * (1-weight_prob)
    return m

In [6]:
def one_slot_greedy_T(state):
    B1, B2, rem1, rem2, h1, h2, wt1, wt2 = state[0], state[1], state[2], state[3], state[4], state[5], state[6], state[7]
    rho1 = cp.Variable(nonneg=True)
    rho2 = cp.Variable(nonneg=True)
    P1 = cp.Variable(nonneg=True)
    P2 = cp.Variable(nonneg=True)
    
    obj_fn =  wt1 * cp.exp(-(rmax - (rem1 - rho1)) ) + wt2 * cp.exp(-(rmax - (rem2 - rho2)) )

    constraints = [P1 >= 0, rho1 >= 0, P2 >= 0, rho2 >=0, P1 <= B1, P2 <= B2, rho1 <= rem1, rho2 <= rem2]
    #MAC Constraints
    constraints.append( rho1 <= cp.log(1 + (h1*P1) ) * cp.inv_pos(cp.log(2))  )
    constraints.append( rho2 <= cp.log(1 + (h2*P2) ) * cp.inv_pos(cp.log(2))  )
    constraints.append(rho1+rho2 <= cp.log(1+ h1*P1 + h2*P2) * cp.inv_pos(cp.log(2)) )
    #Other
    # constraints.append(P1 >= (cp.exp(rem1 * cp.log(2)) -1)/h1)
    # constraints.append(P2 >= (cp.exp(rem2* cp.log(2)) -1)/h2)
    objective =cp.Minimize(obj_fn)
    prob = cp.Problem(objective, constraints)
    # optimal_val = prob.solve(solver=cp.SCS, eps=1e-8, verbose =False)
    optimal_val = prob.solve(verbose =False)
    return optimal_val

In [7]:
V = []
for state in all_states:
        V.append(one_slot_greedy_T(state))
V = np.array(V)
V= np.abs(V)
print(V.shape)

/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


(1296,)


In [8]:
def quad_fit(all_states, Vy):
    X  = all_states
    y = Vy
    y = y.reshape(-1, 1)
    n = X.shape[1]
    P = cp.Variable((n, n), symmetric=True)
    q = cp.Variable(n)
    r = cp.Variable()
    temp = [cp.quad_form(X[i, :], P) + X[i, :] @ q + r for i in range(X.shape[0])]
    temp = cp.vstack(temp)
    objective = cp.Minimize(cp.sum_squares(temp - y))
    constraints = [P >> 0, temp >= 1e-8]
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, eps=1e-8)
    P_opt = P.value
    q_opt = q.value
    r_opt = r.value
    def is_positive_semidefinite(matrix):
        # Check if the matrix is symmetric
        if not np.allclose(matrix, matrix.T):
            print("The matrix is not symmetric.")
            return False
        eigenvalues = np.linalg.eigvals(matrix)
        if np.all(eigenvalues >= 0):
            return True
        else:
            print("The matrix has negative eigenvalues:", eigenvalues)
            return False
    flag = 0
    if is_positive_semidefinite(P_opt):
        print("The matrix is positive semidefinite.")
    else:
        print("The matrix is not positive semidefinite.")
        flag = 1
        # print(P_opt)
    return P_opt, q_opt, r_opt, flag

In [9]:
def one_slot_greedy_2(state, P_opt, q_opt, r_opt, gamma, send_end):
    B1, B2, rem1, rem2, h1, h2, wt1, wt2 = state[0], state[1], state[2], state[3], state[4], state[5], state[6], state[7]
    rho1 = cp.Variable(nonneg=True)
    rho2 = cp.Variable(nonneg=True)
    P1 = cp.Variable(nonneg=True)
    P2 = cp.Variable(nonneg=True)
    obj_fn = wt1 * cp.exp(-(rmax - (rem1 - rho1))) + wt2 * cp.exp(-(rmax - (rem2 - rho2)))
    B_end1, B_end2  = B1 - P1, B2 - P2
    B_ch_arr1 = [B_end1, B_end1+1]
    B_ch_arr2 = [B_end2, B_end2+1]
    r_ch_arr1 = [rem1 - rho1, rmax]
    r_ch_arr2 = [rem2 - rho2, rmax]
    wt_ch_arr1, wt_ch_arr2 = [1, 2], [1, 2]
    all_possible_next_states = []
    for Bch1 in B_ch_arr1:
        for Bch2 in B_ch_arr2:
            for rch1 in r_ch_arr1:
                for rch2 in r_ch_arr2:
                        for h1 in h_vals_u1:
                            for h2 in h_vals_u1:
                                for w1 in wt_ch_arr1:
                                    for w2 in wt_ch_arr2:
                                        all_possible_next_states.append([Bch1,Bch2,rch1,rch2, h1,h2, w1, w2])
    ns = np.array(all_possible_next_states)
    Vz = []
    for next_state in all_possible_next_states:
        ns = cp.hstack(next_state)
        value = cp.quad_form(ns, P_opt) + ns @ q_opt + r_opt
        Vz.append(value)
    Vz = cp.hstack(Vz)
    B_ch_pr = np.array([(1 - e_prob), e_prob])
    r_ch_pr = np.array([(1 - pkt_prob), pkt_prob])
    if wt1 == 1:
        wt_ch_pr1 =[st_tr_wt(1, 1), st_tr_wt(1, 2)]
    else:
        wt_ch_pr1 =[st_tr_wt(2, 1), st_tr_wt(2, 2)]
    if wt2 == 1:
        wt_ch_pr2 =[st_tr_wt(1, 1), st_tr_wt(1, 2)]
    else:
        wt_ch_pr2 =[st_tr_wt(2, 1), st_tr_wt(2, 2)]
    next_state_prob = []
    for b_pr in B_ch_pr:
        for b_pr2 in B_ch_pr:
            for r_pr in r_ch_pr:
                for r_pr2 in r_ch_pr:
                    for h_pr in h1_prob:
                        for h_pr2 in h1_prob:
                            for wt_pr in wt_ch_pr1:
                                for wt_pr2 in wt_ch_pr2:
                                    next_state_prob.append(b_pr * b_pr2 * r_pr * r_pr2 * h_pr * h_pr2 * wt_pr * wt_pr2)                     
    next_state_prob = np.array(next_state_prob)
    next_state_prob = cp.hstack(next_state_prob)
    Expected_V = gamma * Vz @ next_state_prob
    obj_fn += Expected_V
    constraints = [P1 >= 0, rho1 >= 0, P2 >= 0, rho2 >=0,  P1 <= B1, P2 <= B2, rho1 <= rem1,rho2 <= rem2 ]
    
    #MAC Constraints
    constraints.append( rho1 <= cp.log(1 + (h1*P1) ) * cp.inv_pos(cp.log(2)) )
    constraints.append( rho2 <= cp.log(1 + (h2*P2) ) * cp.inv_pos(cp.log(2)) )
    constraints.append(rho1+rho2 <= cp.log(1+ h1*P1 + h2*P2) * cp.inv_pos(cp.log(2)) )

    #Other
    # constraints.append(P1 >= (cp.exp(rem1 * cp.log(2)) -1)/h1)
    # constraints.append(P2 >= (cp.exp(rem2* cp.log(2)) -1)/h2)

    objective = cp.Minimize(obj_fn)
    prob = cp.Problem(objective, constraints)
    # optimal_val = prob.solve(solver=cp.SCS, eps=1e-8, verbose =False)
    optimal_val = prob.solve(verbose =False)
    send_end.send(optimal_val)
    # return optimal_val

In [10]:
def one_slot_greedy_33(state, P_opt, q_opt, r_opt, gamma=0.99):
    B1, B2, rem1, rem2, h1, h2, wt1, wt2 = state[0], state[1], state[2], state[3], state[4], state[5], state[6], state[7]
    rho1 = cp.Variable(nonneg=True)
    rho2 = cp.Variable(nonneg=True)
    P1 = cp.Variable(nonneg=True)
    P2 = cp.Variable(nonneg=True)
    obj_fn = wt1 * cp.exp(-(rmax - (rem1 - rho1))) + wt2 * cp.exp(-(rmax - (rem2 - rho2)))
    B_end1, B_end2  = B1 - P1, B2 - P2
    B_ch_arr1 = [B_end1, B_end1+1]
    B_ch_arr2 = [B_end2, B_end2+1]
    r_ch_arr1 = [rem1 - rho1, rmax]
    r_ch_arr2 = [rem2 - rho2, rmax]
    wt_ch_arr1, wt_ch_arr2 = [1, 2], [1, 2]
    all_possible_next_states = []
    for Bch1 in B_ch_arr1:
        for Bch2 in B_ch_arr2:
            for rch1 in r_ch_arr1:
                for rch2 in r_ch_arr2:
                        for h1 in h_vals_u1:
                            for h2 in h_vals_u1:
                                for w1 in wt_ch_arr1:
                                    for w2 in wt_ch_arr2:
                                        all_possible_next_states.append([Bch1,Bch2,rch1,rch2, h1,h2, w1, w2])
    ns = np.array(all_possible_next_states)
    Vz = []
    for next_state in all_possible_next_states:
        ns = cp.hstack(next_state)
        value = cp.quad_form(ns, P_opt) + ns @ q_opt + r_opt
        Vz.append(value)
    Vz = cp.hstack(Vz)
    B_ch_pr = np.array([(1 - e_prob), e_prob])
    r_ch_pr = np.array([(1 - pkt_prob), pkt_prob])
    if wt1 == 1:
        wt_ch_pr1 =[st_tr_wt(1, 1), st_tr_wt(1, 2)]
    else:
        wt_ch_pr1 =[st_tr_wt(2, 1), st_tr_wt(2, 2)]

    if wt2 == 1:
        wt_ch_pr2 =[st_tr_wt(1, 1), st_tr_wt(1, 2)]
    else:
        wt_ch_pr2 =[st_tr_wt(2, 1), st_tr_wt(2, 2)]
    next_state_prob = []
    for b_pr in B_ch_pr:
        for b_pr2 in B_ch_pr:
            for r_pr in r_ch_pr:
                for r_pr2 in r_ch_pr:
                    for h_pr in h1_prob:
                        for h_pr2 in h1_prob:
                            for wt_pr in wt_ch_pr1:
                                for wt_pr2 in wt_ch_pr2:
                                    next_state_prob.append(b_pr * b_pr2 * r_pr * r_pr2 * h_pr * h_pr2 * wt_pr * wt_pr2)
    next_state_prob = cp.hstack(next_state_prob)
    Expected_V = gamma * Vz @ next_state_prob
    obj_fn += Expected_V
    constraints = [P1 >= 0, rho1 >= 0, P2 >= 0, rho2 >=0,  P1 <= B1, P2 <= B2, rho1 <= rem1,rho2 <= rem2 ]
    #MAC constraints
    constraints.append( rho1 <= cp.log(1 + (h1*P1) ) * cp.inv_pos(cp.log(2)) )
    constraints.append( rho2 <= cp.log(1 + (h2*P2) ) * cp.inv_pos(cp.log(2)) )
    constraints.append(rho1+rho2 <= cp.log(1+ h1*P1 + h2*P2) * cp.inv_pos(cp.log(2)) )

    #Other
    # constraints.append(P1 >= (cp.exp(rem1 * cp.log(2)) -1)/h1)
    # constraints.append(P2 >= (cp.exp(rem2* cp.log(2)) -1)/h2)
    
    objective = cp.Minimize(obj_fn)
    prob = cp.Problem(objective, constraints)
    # optimal_val = prob.solve(solver=cp.SCS, eps=1e-8, verbose =False)
    optimal_val = prob.solve(verbose =False)
    P1, P2, rho1, rho2 = P1.value, P2.value, rho1.value, rho2.value

    if np.round(P1, 3) > np.round(B1, 3):
        print('error1', np.round(P1, 3), np.round(B1, 3))
    if np.round(P2, 3) > np.round(B2, 3):
        print('error1b', np.round(P2, 3), np.round(B2, 3))
    if np.round(rho1,3) > np.round(rem1,3):
        print('error 2', np.round((rho1, rem1), 3))
    if np.round(rho2,3) > np.round(rem2,3):
        print('error 2b', np.round((rho2, rem2), 3))
    if np.round(rho1,3) > np.round(np.log2(1 + P1 * h1), 3):
        print('error3', np.round((rho1, np.log2(1+P1*h1)), 3))
    if np.round(rho2,3) > np.round(np.log2(1 + P2 * h2), 3):
        print('error3b', np.round((rho1, np.log2(1+P2*h2)), 3))
    if np.round(rho1+rho2,3) > np.round(np.log2(1 +P1 * h1+ P2 * h2), 3):
        print('error 4', np.round((rho1+rho2, np.log2(1 +P1 * h1+ P2 * h2)), 3))
    return P1, P2, rho1, rho2, optimal_val

In [11]:
def policy_run(T_horizon,P_opt, q_opt, r_opt):
    M = 2
    tot_dist = 0
    B_prev1 = 1
    B_prev2 = 1
    rem_prev1 = 0
    rem_prev2 = 0
    wt_start1 = 1
    wt_start2 = 1
    for t in range(T_horizon):
        wt1 = np.random.choice([1, 2], p = [weight_prob, (1-weight_prob)])
        wt2 = np.random.choice([1, 2], p = [weight_prob, (1-weight_prob)])
        h1= np.random.choice([0.1, 1], p = h1_prob)
        h2= np.random.choice([0.1, 1], p = h1_prob)
        pkt1 = np.random.choice([1, 0], p=[pkt_prob, (1 - pkt_prob)])
        pkt2 = np.random.choice([1, 0], p=[pkt_prob, (1 - pkt_prob)])
        E1 = np.random.choice([1, 0], p=[ e_prob, (1-e_prob)])
        E2 = np.random.choice([1, 0], p=[ e_prob, (1-e_prob)])
        if pkt1 == 1:
            wt_start1 = wt1
            rem_start1 = rmax
        else:
            rem_start1 = rem_prev1
        if pkt2 == 1:
            wt_start2 = wt2
            rem_start2 = rmax
        else:
            rem_start2 = rem_prev2
        B_start1 = np.minimum(B_prev1+E1, B_max)
        B_start2 = np.minimum(B_prev2+E2, B_max)
        state_slot = np.array([B_start1, B_start2, rem_start1, rem_start2, h1, h2, wt_start1, wt_start2])
        t12 =time.time()
        P1, P2, rho1, rho2, VT = one_slot_greedy_33(state_slot, P_opt, q_opt, r_opt, 0.99)
        print('time taken', time.time()-t12)
   
        B_prev1 = B_start1 - P1
        B_prev2 = B_start2 - P2
        rem_prev1 = rem_start1 - rho1
        rem_prev2 = rem_start2 - rho2
        B_prev1 = np.maximum(B_prev1, 0)
        B_prev2 = np.maximum(B_prev2, 0)
        rem_prev1 = np.maximum(rem_prev1, 0)
        rem_prev2 = np.maximum(rem_prev2, 0)
        dist = wt_start1 * np.exp(-(rmax - rem_prev1)) + wt_start2 * np.exp(-(rmax - rem_prev2))
        tot_dist+=  dist
        if t%100 == 0:
            print('obj', t, tot_dist/((t+1)*M))
        # res = tot_dist/(T_horizon*M)
        # print('final objective value', pkt_prob, e_prob, weight_prob, res)
        # print('-----------------------------------')

In [12]:
T = 11
stp = 1
import time
while(stp < T):
    t11 = time.time()
    print('stp', stp)
    P_opt, q_opt, r_opt, flag = quad_fit(all_states, V)
    def make_positive_semidefinite(matrix, tolerance=1e-10):
        matrix = (matrix + matrix.T) / 2
        eigenvalues, eigenvectors = np.linalg.eigh(matrix)
        eigenvalues[eigenvalues < tolerance] = 0
        perturbed_matrix = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
        perturbed_matrix = (perturbed_matrix + perturbed_matrix.T) / 2
        return perturbed_matrix
    # Check if the perturbed matrix is PSD
    def is_positive_semidefinite(matrix):
        eigenvalues = np.linalg.eigvals(matrix)
        return np.all(eigenvalues >= 0)    
    if flag == 1:
        psd_matrix = make_positive_semidefinite(P_opt)
        P_opt = psd_matrix
        # print("\nPerturbed matrix:")
        # print(psd_matrix)
        if is_positive_semidefinite(psd_matrix):
            print("\nThe perturbed matrix is positive semidefinite.")
        else:
            print("\nThe perturbed matrix is not positive semidefinite.")
    # policy_run(100,P_opt, q_opt, r_opt)
    psd_matrix = make_positive_semidefinite(P_opt)
    P_opt = psd_matrix
    V_next = np.array([])
    z = 18
    q = 0
    while(z <= len(all_states)):
        t2 = time.time()
        jobs = []
        pipe_list = []
        for state in all_states[q:z]:
            recv_end, send_end = multiprocessing.Pipe(False)
            p = Process(target=one_slot_greedy_2, args=(state, P_opt, q_opt, r_opt, 0.99, send_end))
            jobs.append(p)
            pipe_list.append(recv_end)
        for process in jobs:
            process.start()
        for process in jobs:
            process.join()
        V_next_1 = np.array([x.recv() for x in pipe_list])
        V_next = np.concatenate((V_next, V_next_1))
        q = z
        z+= 18
        if z%144 == 0:
            print('z', z)
        # print(time.time() - t2)
    V_next = np.abs(V_next)
    V  = V_next
    print(V)
    print('-----------------------')
    print('time per itr', time.time() - t11)
    stp+= 1
print('value iteration done', pkt_prob, e_prob, weight_prob)

stp 1


/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:167: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn("Objective contains too many subexpressions. "
/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:173: UserWarning: Constraint #1 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn(f"Constraint #{i} contains too many subexpressions. "
/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:173: UserWarning: Constraint #0 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn(f"Constraint #{i} contains too many subexpressions. "
/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:173: UserWarning: Constraint #2 contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compil

The matrix is positive semidefinite.
z 144
z 288
z 432
z 576
z 720
z 864
z 1008
z 1152
z 1296
[2.19718905 2.85279    2.85279    ... 2.64291798 2.64291798 3.27650444]
-----------------------
time per itr 206.00336575508118
stp 2
The matrix is positive semidefinite.


/home/sysad/.local/lib/python3.10/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


z 144
z 288
z 432
z 576
z 720
z 864
z 1008
z 1152


KeyboardInterrupt: 

In [ ]:
policy_run(2000,P_opt, q_opt, r_opt)